## 1. Imports & Setup

In [1]:
import os
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    auc
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

## 2. Paths

In [2]:
BASE = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROC = os.path.join(BASE, "data", "processed")
MODEL_DIR = os.path.join(BASE, "models")

os.makedirs(MODEL_DIR, exist_ok=True)

## 3. Load Data

In [4]:
def load_fraud():
    path = os.path.join(PROC, "fraud")

    train = pd.read_csv(os.path.join(path, "04_train.csv"))
    test  = pd.read_csv(os.path.join(path, "05_test.csv"))

    X_train = train.drop(columns=["class"])
    y_train = train["class"]

    X_test = test.drop(columns=["class"])
    y_test = test["class"]

    return X_train, y_train, X_test, y_test


def load_credit():
    path = os.path.join(PROC, "creditcard")

    train = pd.read_csv(os.path.join(path, "02_train.csv"))
    test  = pd.read_csv(os.path.join(path, "03_test.csv"))

    X_train = train.drop(columns=["Class"])
    y_train = train["Class"]

    X_test = test.drop(columns=["Class"])
    y_test = test["Class"]

    return X_train, y_train, X_test, y_test

X_tr_f, y_tr_f, X_te_f, y_te_f = load_fraud()
X_tr_cc, y_tr_cc, X_te_cc, y_te_cc = load_credit()

## 4. Evaluation Function

In [5]:
def evaluate(model, X_test, y_test, name):
    proba = model.predict_proba(X_test)[:, 1]

    best_thr, best_f1 = 0.5, 0

    for t in np.arange(0.1, 0.95, 0.05):
        preds = (proba >= t).astype(int)
        f1 = f1_score(y_test, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    preds = (proba >= best_thr).astype(int)

    precision, recall, _ = precision_recall_curve(y_test, proba)
    auc_pr = auc(recall, precision)
    roc = roc_auc_score(y_test, proba)

    cm = confusion_matrix(y_test, preds)

    print(f"\n{name}")
    print("Best Threshold:", round(best_thr, 2))
    print("AUC-PR:", round(auc_pr, 4))
    print("ROC-AUC:", round(roc, 4))
    print("F1:", round(best_f1, 4))
    print(cm)
    print(classification_report(y_test, preds))

## 5. Logistic Regression

In [6]:
lr_f = LogisticRegression(max_iter=2000, class_weight="balanced")
lr_f.fit(X_tr_f, y_tr_f)

evaluate(lr_f, X_te_f, y_te_f, "Logistic Regression - Fraud")

pickle.dump(lr_f, open(os.path.join(MODEL_DIR, "lr_fraud.pkl"), "wb"))


Logistic Regression - Fraud
Best Threshold: 0.8
AUC-PR: 0.623
ROC-AUC: 0.7676
F1: 0.6901
[[27393     0]
 [ 1339  1491]]
              precision    recall  f1-score   support

           0       0.95      1.00      0.98     27393
           1       1.00      0.53      0.69      2830

    accuracy                           0.96     30223
   macro avg       0.98      0.76      0.83     30223
weighted avg       0.96      0.96      0.95     30223



In [7]:
lr_cc = LogisticRegression(max_iter=2000, class_weight="balanced")
lr_cc.fit(X_tr_cc, y_tr_cc)

evaluate(lr_cc, X_te_cc, y_te_cc, "Logistic Regression - CreditCard")

pickle.dump(lr_cc, open(os.path.join(MODEL_DIR, "lr_cc.pkl"), "wb"))


Logistic Regression - CreditCard
Best Threshold: 0.9
AUC-PR: 0.7048
ROC-AUC: 0.9658
F1: 0.398
[[56428   223]
 [   16    79]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.26      0.83      0.40        95

    accuracy                           1.00     56746
   macro avg       0.63      0.91      0.70     56746
weighted avg       1.00      1.00      1.00     56746



## 6. XGBoost Model

In [8]:
def train_xgb(X_train, y_train, X_test, y_test, name):

    spw = (y_train == 0).sum() / (y_train == 1).sum()

    model = XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        scale_pos_weight=spw,
        eval_metric="aucpr",
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    evaluate(model, X_test, y_test, name)

    return model

## 7. Train XGBoost

In [9]:
xgb_f = train_xgb(X_tr_f, y_tr_f, X_te_f, y_te_f, "XGBoost - Fraud")
pickle.dump(xgb_f, open(os.path.join(MODEL_DIR, "xgb_fraud.pkl"), "wb"))


XGBoost - Fraud
Best Threshold: 0.75
AUC-PR: 0.6179
ROC-AUC: 0.7598
F1: 0.6901
[[27393     0]
 [ 1339  1491]]
              precision    recall  f1-score   support

           0       0.95      1.00      0.98     27393
           1       1.00      0.53      0.69      2830

    accuracy                           0.96     30223
   macro avg       0.98      0.76      0.83     30223
weighted avg       0.96      0.96      0.95     30223



In [10]:
xgb_cc = train_xgb(X_tr_cc, y_tr_cc, X_te_cc, y_te_cc, "XGBoost - CreditCard")
pickle.dump(xgb_cc, open(os.path.join(MODEL_DIR, "xgb_cc.pkl"), "wb"))


XGBoost - CreditCard
Best Threshold: 0.5
AUC-PR: 0.8248
ROC-AUC: 0.979
F1: 0.8736
[[56648     3]
 [   19    76]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.96      0.80      0.87        95

    accuracy                           1.00     56746
   macro avg       0.98      0.90      0.94     56746
weighted avg       1.00      1.00      1.00     56746



## 8. Model Comparison

In [11]:
results = pd.DataFrame([
    ["LR Fraud", "Logistic Regression", "Fraud"],
    ["LR CC", "Logistic Regression", "CreditCard"],
    ["XGB Fraud", "XGBoost", "Fraud"],
    ["XGB CC", "XGBoost", "CreditCard"]
], columns=["Model", "Type", "Dataset"])

results

,Model,Type,Dataset
0,LR Fraud,Logistic Regression,Fraud
1,LR CC,Logistic Regression,CreditCard
2,XGB Fraud,XGBoost,Fraud
3,XGB CC,XGBoost,CreditCard


In [12]:
print("Pipeline Complete ✔")

Pipeline Complete ✔
